# Análisis Exploratorio de Datos (EDA)
### Ciencia de Datos · 7.º semestre · Carrera de Sistemas

**Caso de estudio:** el catálogo de una librería en línea (`book1.csv`, 1 000 libros)

> *"El análisis exploratorio es trabajo de detectives: buscar pistas antes de acusar a un modelo."* — inspirado en John W. Tukey (1977)



## Objetivos de aprendizaje

Al terminar la clase el estudiante será capaz de:

1. Formular el **contexto y las preguntas** que guían un análisis.
2. **Cargar, inspeccionar y describir** un dataset (dimensiones, variables, tipos).
3. **Evaluar la calidad** de los datos y documentar decisiones de limpieza.
4. Calcular e **interpretar estadísticas** descriptivas (centro, dispersión, forma, correlación).
5. **Agrupar y resumir** para responder preguntas de negocio.
6. **Visualizar** eligiendo el gráfico correcto y **comunicar hallazgos** con sus límites.

## Agenda (13 etapas)

| # | Etapa | Pregunta que responde |
|---|-------|-----------------------|
| 1 | Contexto del problema | ¿Para qué analizamos? |
| 2 | Conociendo el dataset | ¿Qué es cada columna? |
| 3 | Cargar los datos | ¿Cómo entran los datos a Python? |
| 4 | Primer vistazo | ¿Cómo se ven? |
| 5 | Dimensiones | ¿Cuántos son? |
| 6 | Variables | ¿De qué naturaleza es cada una? |
| 7 | Tipos de datos | ¿Python las entiende bien? |
| 8 | Calidad de los datos | ¿Puedo confiar en ellos? |
| 9 | Estadísticas básicas | ¿Cómo se comportan? |
| 10 | Agrupaciones | ¿Cómo cambian por segmento? |
| 11 | Preguntas de negocio | ¿Qué decisiones apoyan? |
| 12 | Visualización | ¿Cómo lo muestro? |
| 13 | Interpretación | ¿Qué concluyo y con qué límites? |



## ¿Qué es el EDA y por qué importa?

**EDA (Exploratory Data Analysis)** = conjunto de técnicas para **entender los datos antes de modelar**: resumirlos, visualizarlos, detectar problemas y generar hipótesis.

- 🧹 Descubre **errores** y **valores raros** antes de que contaminen un modelo.
- 🧭 Orienta **qué preguntas** tienen sentido y qué variables usar.
- 🧪 Genera **hipótesis** que luego se validan (estadística inferencial / ML).
- 🗣️ Permite **explicar** los datos a personas no técnicas.

> ⚠️ **Garbage in, garbage out**: un modelo sofisticado sobre datos sucios produce resultados sofisticadamente equivocados.

**Ciclo típico:** Pregunta → Datos → Limpieza → Exploración → Hallazgos → Nuevas preguntas 🔁

In [1]:
# ── Configuración general ─────────────────────────────────────────────
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RUTA = "../BDD/book1.csv"   # si usas Google Colab: sube el archivo o monta tu Drive y ajusta esta ruta
print("pandas", pd.__version__)

pandas 3.0.6


# 1️⃣ Contexto del problema

##  El escenario

Somos el equipo de datos de **una librería en línea**. La gerencia comercial quiere **ordenar su catálogo** antes de la temporada alta y nos entrega un archivo con **1 000 libros**.

**Lo que la gerencia quiere saber**
- ¿Cómo está compuesto el catálogo (categorías, precios, valoraciones)?
- ¿Los libros más caros están mejor valorados?
- ¿Qué categorías pesan más y cuáles son "nicho"?
- ¿Podemos confiar en los datos que nos dieron?



## Del problema de negocio al problema analítico

| Nivel | Ejemplo en nuestro caso |
|-------|-------------------------|
| **Objetivo de negocio** | Tomar mejores decisiones de surtido y precio del catálogo |
| **Objetivo analítico** | Describir la distribución de precios y valoraciones, y su relación por categoría |
| **Unidad de análisis** | Un libro |
| **Entregable** | Hallazgos claros + limitaciones + recomendaciones |
| **Criterio de éxito** | Que la gerencia pueda decidir *con* el análisis, sabiendo qué tan confiable es |

> 🔑 Sin contexto, el EDA se convierte en "hacer gráficos por hacer".

1. ¿Qué **decisión** cambiaría si descubrimos que el precio *no* influye en la valoración?
2. ¿Qué información **necesitaríamos** (y no tenemos) para hablar de *ventas* o *rentabilidad*?

# 2️⃣ Conociendo el dataset

## 📖 Diccionario de datos

| Columna | Significado | Tipo esperado |
|---------|-------------|---------------|
| `Titulo` | Nombre del libro | Texto |
| `Rating` | Valoración en estrellas (1 a 5) | Entero ordinal |
| `Precio` | Precio de venta (libras £ en el sitio original) | Decimal |
| `Stock` | Unidades disponibles | Entero |
| `URL` | Enlace a la ficha del libro | Texto (identificador) |
| `Descripcion` | Sinopsis del libro | Texto libre |
| `Categoria` | Género o categoría | Categórica nominal |

**Granularidad:** 1 fila = 1 libro. **Fuente:** scraping. **Momento:** una sola "foto" (*snapshot*), sin fechas.

## ✍️ Buenas prácticas antes de tocar el código

- Escribe el **diccionario de datos** (si no existe, créalo mientras exploras).
- Identifica la **unidad de observación** y la **llave** (¿qué columna identifica cada fila?).
- Anota **cómo se obtuvieron** los datos: la fuente determina los sesgos posibles.
- Define **expectativas**: ¿qué rangos y valores son razonables? (así reconoces lo raro).

> 🧠 *Nuestra expectativa:* precios positivos, ratings del 1 al 5, stock ≥ 0, una URL distinta por libro.

# 3️⃣ Cargar los datos

## `pd.read_csv()` y sus parámetros clave

| Parámetro | Para qué sirve |
|-----------|----------------|
| `sep` | Separador (`,` `;` `\t`) |
| `encoding` | Codificación (`utf-8`, `latin-1`) — evita "acentos raros" |
| `usecols` | Cargar solo algunas columnas |
| `nrows` | Cargar solo las primeras *n* filas (archivos enormes) |
| `dtype` | Forzar tipos desde la carga |
| `parse_dates` | Convertir columnas a fecha |
| `na_values` | Qué textos cuentan como nulo |

In [2]:
df_raw = pd.read_csv(RUTA)          # dataset original: NO lo modificaremos
df = df_raw.copy()                  # copia de trabajo
print("✅ Datos cargados:", df.shape)

✅ Datos cargados: (1000, 7)


### Cargar solo lo necesario (útil con archivos grandes)

In [3]:
muestra = pd.read_csv(RUTA, usecols=["Titulo", "Rating", "Precio"], nrows=5)
muestra

,Titulo,Rating,Precio
0,A Light in the Attic,3,51.77
1,Tipping the Velvet,1,53.74
2,Soumission,1,50.10
3,Sharp Objects,4,47.82
4,Sapiens: A Brief History of Humankind,5,54.23


> ✅ **Regla de oro:** conserva siempre el **dato crudo** (`df_raw`) intacto y trabaja sobre una copia. Así puedes volver atrás y tu análisis es **reproducible**.

# 4️⃣ Primer vistazo

##  Mirar antes de calcular
`head()` · `tail()` · `sample()` · `info()`

In [4]:
df.head()

,Titulo,Rating,Precio,Stock,URL,Descripcion,Categoria
0,A Light in the Attic,3,51.77,0,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/ind...,It's hard to imagine a world without A Light in the Attic. This no...,Poetry
1,Tipping the Velvet,1,53.74,0,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index....,"""Erotic and absorbing...Written with starling power.""--""The New Yo...",Historical Fiction
2,Soumission,1,50.10,0,https://books.toscrape.com/catalogue/soumission_998/index.html,"Dans une France assez proche de la nôtre, un homme s’engage dans l...",Fiction
3,Sharp Objects,4,47.82,0,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,"WICKED above her hipbone, GIRL across her heart Words are like a r...",Mystery
4,Sapiens: A Brief History of Humankind,5,54.23,0,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-hu...,From a renowned historian comes a groundbreaking narrative of huma...,History


In [5]:
df.sample(5, random_state=42)     # random_state = resultado reproducible

,Titulo,Rating,Precio,Stock,URL,Descripcion,Categoria
521,"Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--A...",5,11.38,0,https://books.toscrape.com/catalogue/naturally-lean-125-nourishing...,"When you're hungry for a filling meal, it's easy to go for the unh...",Food and Drink
737,The Maze Runner (The Maze Runner #1),1,20.93,0,https://books.toscrape.com/catalogue/the-maze-runner-the-maze-runn...,"If you ain’t scared, you ain’t human.When Thomas wakes up in the l...",Default
740,The Lean Startup: How Today's Entrepreneurs Use Continuous Innovat...,3,33.92,0,https://books.toscrape.com/catalogue/the-lean-startup-how-todays-e...,Most startups fail. But many of those failures are preventable. Th...,Business
660,Shopaholic Ties the Knot (Shopaholic #3),5,48.39,0,https://books.toscrape.com/catalogue/shopaholic-ties-the-knot-shop...,Life has been good for Becky Bloomwood: She’s become the best pers...,Womens Fiction
411,Cravings: Recipes for What You Want to Eat,3,20.50,0,https://books.toscrape.com/catalogue/cravings-recipes-for-what-you...,Though she is best known as a supermodel and Sports Illustrated sw...,Food and Drink


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Titulo       1000 non-null   str    
 1   Rating       1000 non-null   int64  
 2   Precio       1000 non-null   float64
 3   Stock        1000 non-null   int64  
 4   URL          1000 non-null   str    
 5   Descripcion  1000 non-null   str    
 6   Categoria    1000 non-null   str    
dtypes: float64(1), int64(2), str(4)
memory usage: 54.8 KB


###  ¿Qué observamos?
- 7 columnas; **ninguna** tiene nulos según `info()` (¡ojo! *no nulo* no significa *correcto*).
- `Descripcion` parece tener **texto repetido** y termina en `...more`.
- Hay categorías con nombres raros como `Default` o `Add a comment`.
- `Stock` aparece siempre como `0`.



# 5️⃣ Dimensiones

In [7]:
filas, columnas = df.shape
print(f"Filas (observaciones): {filas:,}")
print(f"Columnas (variables):  {columnas}")
print(f"Celdas totales:        {df.size:,}")

Filas (observaciones): 1,000
Columnas (variables):  7
Celdas totales:        7,000


In [8]:
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria usada: {memoria_mb:.2f} MB")
df.memory_usage(deep=True).sort_values(ascending=False)

Memoria usada: 2.56 MB


Descripcion    2355305
URL             146115
Titulo           96842
Categoria        66516
Rating            8000
Precio            8000
Stock             8000
Index              132
dtype: int64

### ¿Y la llave del dataset?
Una **llave** identifica de forma única cada fila. Comprobémoslo:

In [9]:
print("¿URL es única?      ", df["URL"].is_unique)
print("¿Titulo es único?   ", df["Titulo"].is_unique)
df.nunique().sort_values(ascending=False)

¿URL es única?       True
¿Titulo es único?    False


URL            1000
Descripcion     999
Titulo          999
Precio          903
Categoria        50
Rating            5
Stock             1
dtype: int64

- `URL` es única → **buena llave**.
- `Titulo` tiene **1 repetido** → no sirve como llave (veremos por qué en calidad).
- `Stock` tiene **1 solo valor único** → sospechosa (¿variable constante?).

📏 *Regla práctica:* con ~1 000 filas alcanza para describir; para modelar con muchas variables necesitarás más datos.

# 6️⃣ Variables

## Tipos de variables (más allá de "número" o "texto")

```
Variable
├── Cuantitativa (numérica)
│   ├── Continua   → puede tomar cualquier valor en un intervalo   (Precio)
│   └── Discreta   → valores enteros contables                     (Stock)
└── Cualitativa (categórica)
    ├── Nominal    → categorías sin orden                          (Categoria)
    └── Ordinal    → categorías con orden                          (Rating)
+ Especiales: identificador (URL) · texto libre (Titulo, Descripcion)
```

> ⚠️ **Trampa clásica:** `Rating` es un *número*, pero es **ordinal**. La distancia entre 1★ y 2★ no tiene por qué ser igual a la de 4★ y 5★. Por eso **la media de un rating hay que interpretarla con cuidado**.

In [10]:
clasificacion = pd.DataFrame({
    "columna": df.columns,
    "tipo_de_variable": [
        "texto libre", "categórica ordinal", "cuantitativa continua",
        "cuantitativa discreta", "identificador", "texto libre", "categórica nominal",
    ],
    "valores_unicos": df.nunique().values,
    "ejemplo": [str(df[c].iloc[0])[:35] for c in df.columns],
})
clasificacion

,columna,tipo_de_variable,valores_unicos,ejemplo
0,Titulo,texto libre,999,A Light in the Attic
1,Rating,categórica ordinal,5,3
2,Precio,cuantitativa continua,903,51.77
3,Stock,cuantitativa discreta,1,0
4,URL,identificador,1000,https://books.toscrape.com/catalogu
5,Descripcion,texto libre,999,It's hard to imagine a world withou
6,Categoria,categórica nominal,50,Poetry


### 🔧 Una ayuda automática (pero con criterio humano)
La **cardinalidad** (`nunique`) ayuda a decidir: pocas categorías → categórica; casi tantas como filas → identificador o texto.

In [11]:
for col in df.columns:
    n = df[col].nunique()
    print(f"{col:<12} únicos={n:>5}  ({n/len(df):>6.1%} de las filas)")

Titulo       únicos=  999  ( 99.9% de las filas)
Rating       únicos=    5  (  0.5% de las filas)
Precio       únicos=  903  ( 90.3% de las filas)
Stock        únicos=    1  (  0.1% de las filas)
URL          únicos= 1000  (100.0% de las filas)
Descripcion  únicos=  999  ( 99.9% de las filas)
Categoria    únicos=   50  (  5.0% de las filas)


# 7️⃣ Tipos de datos

In [12]:
df.dtypes

Titulo             str
Rating           int64
Precio         float64
Stock            int64
URL                str
Descripcion        str
Categoria          str
dtype: object

### ¿Por qué importa el `dtype`?
- Determina qué **operaciones** son válidas (no puedes promediar texto).
- Afecta la **memoria** y la **velocidad**.
- Un tipo mal inferido suele ser **síntoma de datos sucios** (ej. un precio con `"£"` se lee como texto).

**Conversiones más usadas**

| Necesidad | Herramienta |
|-----------|-------------|
| Cambiar tipo | `astype("float64")` |
| Texto → número (tolerante a errores) | `pd.to_numeric(s, errors="coerce")` |
| Texto → fecha | `pd.to_datetime(s, errors="coerce")` |
| Texto repetitivo → categoría | `astype("category")` |
| Categoría con orden | `pd.CategoricalDtype([...], ordered=True)` |

In [13]:
# to_numeric con errors="coerce": lo inválido se vuelve NaN en lugar de romper el código
s = pd.Series(["12.5", "abc", "7", "£9.99"])
pd.to_numeric(s, errors="coerce")

0   12.50
1     NaN
2    7.00
3     NaN
dtype: float64

In [14]:
# Categoria: pocas etiquetas repetidas → "category" ahorra memoria
antes = df["Categoria"].memory_usage(deep=True)
df["Categoria"] = df["Categoria"].astype("category")
despues = df["Categoria"].memory_usage(deep=True)
print(f"Memoria de Categoria:  antes = {antes/1024:.1f} KB   después = {despues/1024:.1f} KB")

Memoria de Categoria:  antes = 65.1 KB   después = 4.3 KB


In [15]:
# Rating como categoría ORDINAL: respeta el orden 1 < 2 < 3 < 4 < 5
tipo_rating = pd.CategoricalDtype(categories=[1, 2, 3, 4, 5], ordered=True)
df["Rating_cat"] = df["Rating"].astype(tipo_rating)
print(df["Rating_cat"].dtype)
print("¿Un libro de 5★ es 'mayor' que uno de 2★?", df["Rating_cat"].iloc[0] < pd.Series([5], dtype=tipo_rating).iloc[0])

category
¿Un libro de 5★ es 'mayor' que uno de 2★? True


> 💡 Mantenemos `Rating` (número) para cálculos y `Rating_cat` (ordinal) para ordenar y graficar. Elegir la representación correcta es parte del análisis.

# 8️⃣ Calidad de los datos

## Las 5 dimensiones de la calidad de datos

| Dimensión | Pregunta | Prueba en pandas |
|-----------|----------|------------------|
| **Completitud** | ¿Faltan datos? | `isna().sum()` |
| **Unicidad** | ¿Hay duplicados? | `duplicated().sum()` |
| **Validez** | ¿Respeta rangos/formatos? | filtros lógicos, `between()` |
| **Consistencia** | ¿Se contradice o usa etiquetas distintas para lo mismo? | `value_counts()` |
| **Exactitud** | ¿Refleja la realidad? | contraste con la fuente / criterio experto |

Además buscamos **variables constantes**, **valores centinela** (placeholders como `"N/A"`, `"Default"`, `0`) y **texto sucio**.

### 8.1 Completitud — valores nulos

In [16]:
nulos = pd.DataFrame({"nulos": df.isna().sum(), "% nulos": df.isna().mean() * 100})
nulos

,nulos,% nulos
Titulo,0,0.00
Rating,0,0.00
Precio,0,0.00
Stock,0,0.00
URL,0,0.00
Descripcion,0,0.00
Categoria,0,0.00
Rating_cat,0,0.00


✅ Cero nulos… **¿significa que los datos están completos?** No necesariamente: los *valores centinela* esconden la falta de información. Sigamos.

### 8.2 Unicidad — duplicados

In [17]:
print("Filas completamente duplicadas:", df.duplicated().sum())
print("Títulos duplicados:            ", df["Titulo"].duplicated().sum())
df[df["Titulo"].duplicated(keep=False)][["Titulo", "Precio", "Rating", "Categoria", "URL"]]

Filas completamente duplicadas: 0
Títulos duplicados:             1


,Titulo,Precio,Rating,Categoria,URL
236,The Star-Touched Queen,46.02,5,Fantasy,https://books.toscrape.com/catalogue/the-star-touched-queen_764/in...
358,The Star-Touched Queen,32.30,5,Fantasy,https://books.toscrape.com/catalogue/the-star-touched-queen_642/in...


🔎 Mismo título, **distinta URL y distinto precio**: podrían ser **dos ediciones** o un error. *Decisión:* no eliminar (son filas distintas), pero **documentarlo**.

### 8.3 Validez — rangos esperados

In [18]:
print("Precios <= 0:            ", (df["Precio"] <= 0).sum())
print("Ratings fuera de 1-5:    ", (~df["Rating"].between(1, 5)).sum())
print("Rango de precios:        ", df["Precio"].min(), "→", df["Precio"].max())

Precios <= 0:             0
Ratings fuera de 1-5:     0
Rango de precios:         10.0 → 59.99


### 8.4 Variables constantes
Una variable con **un solo valor** no aporta información para comparar ni modelar.

In [19]:
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("Columnas constantes:", constantes)
print(df["Stock"].value_counts())

Columnas constantes: ['Stock']
Stock
0    1000
Name: count, dtype: int64


🚩 **`Stock = 0` en los 1 000 libros**: irreal para una tienda que vende. Lo más probable es un **error de captura del scraping** (el scraper leyó mal la disponibilidad). *Decisión:* **descartar `Stock`** y reportar el problema al equipo que capturó los datos.

### 8.5 Consistencia — valores centinela y categorías "falsas"

In [20]:
df["Categoria"].value_counts().head(8)

Categoria
Default           152
Nonfiction        110
Sequential Art     75
Add a comment      67
Fiction            65
Young Adult        54
Fantasy            48
Romance            35
Name: count, dtype: int64

In [21]:
print("Categorías distintas:", df["Categoria"].nunique())
print("'Default':      ", (df["Categoria"] == "Default").sum())
print("'Add a comment':", (df["Categoria"] == "Add a comment").sum())
print("Descripciones 'Sin descripción disponible':", (df["Descripcion"] == "Sin descripción disponible").sum())

Categorías distintas: 50
'Default':       152
'Add a comment': 67
Descripciones 'Sin descripción disponible': 2


🚩 `Default` y `Add a comment` **no son géneros literarios**: son textos de la página web capturados por error → representan **219 libros (21.9 %) sin categoría real**. Son **valores centinela** que un `isna()` jamás detectaría.

> 🧠 Lección: **mirar los valores más frecuentes** de cada categórica es una de las mejores pruebas de calidad.

🔎 Las categorías con "5.0 de rating" tienen **1 solo libro**: puro ruido. Entre categorías con muestra suficiente, `Travel` (n = 11) y `Fantasy` (n = 48) lideran el precio medio; la de `Fantasy` es **más confiable** por tener más libros.

### `transform`: comparar cada libro contra su grupo

# Preguntas de negocio

## Método para responder una pregunta con datos

1. **Reformula** la pregunta en términos medibles.
2. **Elige** variables y la métrica (¿media? ¿porcentaje? ¿correlación?).
3. **Calcula** (tabla o cifra).
4. **Visualiza** si ayuda.
5. **Concluye** en una frase con su **nivel de confianza y límites**.

> Pregunta vaga → *"¿Los libros caros son mejores?"*  
> Pregunta medible → *"¿El rating medio del cuartil de precios más alto supera al del más bajo?"*

### P1. ¿Qué proporción del catálogo tiene valoración alta (≥ 4★)?

In [22]:
pct_alto = df_limpio["rating_alto"].mean() * 100
print(f"{pct_alto:.1f}% de los libros tiene rating ≥ 4★  ({df_limpio['rating_alto'].sum()} de {len(df_limpio)})")

NameError: name 'df_limpio' is not defined

### P2. ¿Los libros más caros tienen mejor valoración?

In [ ]:
df_limpio.groupby("segmento_precio", observed=True).agg(
    n=("Titulo", "count"),
    rating_medio=("Rating", "mean"),
    pct_rating_alto=("rating_alto", lambda s: s.mean() * 100),
).round(2)

**Conclusión P2:** el rating medio del segmento *Premium* (≈ 3.10) es **prácticamente igual** al del *Económico* (≈ 3.02). **No hay evidencia** de que precio y valoración estén relacionados; cobrar más no garantiza mejor valoración (ni peor).

### P3. ¿La concentración del catálogo? (Principio de Pareto)
¿Cuántas categorías reales acumulan el 80 % de los libros?

In [ ]:
conteo = df_limpio.loc[df_limpio["Categoria"] != "Sin categoría", "Categoria"].value_counts()
acum = conteo.cumsum() / conteo.sum() * 100
n80 = int((acum < 80).sum() + 1)
print(f"{n80} de {len(conteo)} categorías reales ({n80/len(conteo):.0%}) acumulan el 80% de los libros")

🔎 **18 de 48 categorías** (≈ 38 %) concentran el 80 % de los títulos (con 219 libros sin categoría fuera de este cálculo). Hay una **cola larga** de categorías con muy pocos libros: candidatas a consolidarse.

### P4. ¿Descripciones más largas se asocian con mejor valoración?

In [ ]:
corr_desc = df_limpio[["n_palabras", "Rating"]].corr(method="spearman").iloc[0, 1]
print(f"Correlación de Spearman (n_palabras vs Rating): {corr_desc:.3f}")
df_limpio.groupby("Rating")["n_palabras"].median()

**Conclusión P4:** la relación es **nula** (|ρ| ≈ 0). La longitud de la descripción no explica la valoración. *(Nota: la correlación no dice nada de la **calidad** del texto.)*

##  Tu turno

Abre **`02_EDA_Ejercicios.ipynb`** y resuelve los ejercicios aplicando lo visto hoy.  


**Para profundizar**
- Tukey, J. W. (1977). *Exploratory Data Analysis.*
- McKinney, W. *Python for Data Analysis* (autor de pandas).
- Documentación: [pandas](https://pandas.pydata.org/docs/) · [seaborn](https://seaborn.pydata.org/) · [matplotlib](https://matplotlib.org/)